# VoiceLab starter notebook
AIML ZG520. Keep all outputs. Commit as you work (`!python voicelab.py save "message"`). Parts marked [Critical] must be your own code.

First time only: upload the kit and your `<ROLLNO>_config.json`, then run `!python voicelab.py setup`. Check progress any time with `!python voicelab.py progress`.

In [ ]:
# Colab: upload <ROLLNO>_config.json (setup copies it to config.json), voicelab_utils.py and your data/ folder first
import numpy as np, matplotlib.pyplot as plt
from voicelab_utils import *
cfg = load_config('config.json')
set_seed(cfg)
res = Results('results.json')
print(cfg['vocab'])
print(cfg['sentences_spoken'])

## Part 0: Dataset and audit

Three recording blocks: words (30 clips), sentences (4 clips) and room noise (about 11 s). Each block starts with your spoken code, then pauses of 2 s between items.

1. Run the next cell (or `python voicelab.py script`) to print your read-aloud script.
2. Record the three blocks. Save the originals as `data/original/words_block.*`, `sentences_block.*` and `noise_block.*`.
3. Run the cell after that. It converts to 16 kHz mono, splits the clips, audits them and checks your spoken code. Read what it prints: `CHECK` lines name clips to listen to, and `STOPPED` means a block must be recorded again with longer pauses.

In [ ]:
recording_script(cfg)

In [ ]:
# Whole of Part 0 in one command. Run it in a terminal first (it also checks your spoken code with Whisper);
# the options below only repeat the audit when the clips already exist.
!python voicelab.py data --skip-existing --no-whisper

In [ ]:
rows = audit_folder('data')
print_table([r for r in rows if r['clipping_pct'] > 0.5 or r['snr_db'] < 15][:20])  # clips worth a second look

## Part 1: Signal analysis and features
TODO: hand DFT (markdown cell), plot a sentence, [Critical] MFCC pipeline, [Critical] F0 by autocorrelation on your sentences.

In [ ]:
# [starter] plumbing, not counted as your own code
# Setup for Part 1 (plumbing only). Your front-end settings are in cfg['front_end'].
import glob
fe = cfg['front_end']
words = {w: [f'data/words/{w}_{r}.wav' for r in range(3)] for w in cfg['vocab']}   # 3 repetitions of each word
first_clip = sorted(glob.glob('data/words/*.wav'))[0]                               # the clip for 'mfcc_sum_first_clip'
print(fe)
print('first clip:', first_clip)

In [ ]:
# [starter] plumbing, not counted as your own code
# Spectrogram of a sentence at your sampling rate (a quick helper; use your own STFT if you prefer).
x, fs = load_wav('data/sentences/sent_0.wav', sr=fe['sr'])
plot_wave_spec(x, fs, title='sent_0')
savefig('p1_sent0_spectrogram')      # figures/p1_sent0_spectrogram.png goes into your report
plt.show()

In [ ]:
# [Critical] MFCC pipeline: write it yourself with NumPy/SciPy (no librosa.feature.mfcc, no filterbank helper).
# def my_mfcc(x, fs, fe): ...
# mine = my_mfcc(*load_wav(first_clip, sr=fe['sr']), fe)          # shape (frames, n_mfcc)
# Verification only: compare with librosa configured identically, then report the maximum difference per coefficient.
# compare('MFCC vs librosa', mine, librosa_mfcc)
# res.set('mfcc_sum_first_clip', mine.sum())

In [ ]:
# [Critical] F0 by autocorrelation on your sentences: write it yourself.
# savefig('p1_f0_sent0')

In [ ]:
# Save each figure for the report with savefig('name') right after you plot it.
# res.set('mfcc_sum_first_clip', value)  # sum of your own static MFCCs (c0..c_n-1, liftered, no deltas), first clip in data/words

## Part 2: HMM acoustic model
TODO: hand calculation, [Critical] Gaussian emissions, forward, Viterbi, segmental k-means training, evaluation.

In [ ]:
# [starter] plumbing, not counted as your own code
# Setup for Part 2 (plumbing only). Training repetitions and the held-out repetition come from your configuration.
held = cfg['hmm']['heldout_rep']
train_files = {w: [p for r, p in enumerate(words[w]) if r != held] for w in cfg['vocab']}
test_files = {w: words[w][held] for w in cfg['vocab']}
print('held-out repetition:', held)
print('states per word model:', cfg['hmm']['states'])
# [Critical] Gaussian emissions, forward, Viterbi, segmental k-means, evaluation: yours.
# Plot the confusion matrix with matplotlib and call savefig('p2_confusion').

In [ ]:
# res.set('toy_hmm_forward', value); res.set('toy_hmm_viterbi', value)

## Part 3: CTC
TODO: hand calculation, [Critical] CTC forward algorithm, verify with torch.nn.CTCLoss.

In [ ]:
# [Critical] CTC forward algorithm: yours. Check it against torch.nn.CTCLoss (verification only):
# import torch
# loss = torch.nn.CTCLoss(blank=0, reduction='sum')(log_probs, targets, input_lengths, target_lengths)
# compare('CTC forward vs torch', mine_neg_log_prob, loss.item())

In [ ]:
# res.set('toy_ctc_prob', value)

## Part 4: Language model and lexicon
TODO: hand calculation, [Critical] Kneser-Ney bigram, lexicon.

In [ ]:
sents = brown_sentences()   # Brown corpus as lists of lowercase words. Choose and state your own split.
cmu = cmudict_dict()       # CMUdict {word: [phone lists]}; use g2p_en for words it lacks

In [ ]:
# res.set('toy_add1_bigram', value); res.set('toy_kn_bigram', value)

## Part 5: WFST
TODO: hand calculation (determinise, minimise), [Critical] compose() in wfst_starter.py, then L o G for your own words.

In [ ]:
# [starter] plumbing, not counted as your own code
from wfst_starter import FST, compose   # FST container and shortest path are given; compose() is yours to write
# [Critical] compose(): implement it in wfst_starter.py, test on small examples, then build L o G for your own words.
print(cfg['vocab'])

In [ ]:
# res.set('wfst_det_states', value)

## Part 6: Recognisers and decoding
TODO: [Critical] WER, wav2vec2 and Whisper outputs, [Critical] CTC prefix beam search with LM fusion, robustness.

The next cell mixes your sentences with your room noise at each SNR in your configuration (`data/noisy/`) and saves the wav2vec2 log posteriors and Whisper text in `cache/`. It needs internet the first time and takes a few minutes on CPU. Computing WER, grouping the errors, the beam search and the plots are yours.

In [ ]:
!python voicelab.py precompute --if-missing
transcripts = load_transcripts()                    # Whisper text, e.g. transcripts['sent_0_clean']
logp, tokens = load_posteriors('sent_0_clean')      # wav2vec2 log posteriors (T x V) and the token list

In [ ]:
# [starter] plumbing, not counted as your own code
# Names of the cached outputs: clean sentences and each noise level in your configuration.
refs = cfg['sentences_spoken']                                  # reference text for WER
names = [f'sent_{i}_clean' for i in range(len(refs))] + [f'sent_{i}_snr{s}' for i in range(len(refs)) for s in cfg['noise']['snr_db']]
print(names[:6], '...')
print('beam widths:', cfg['decoding']['beam_widths'])
# [Critical] WER (edit-distance DP), CTC prefix beam search with LM fusion, error analysis: yours.
# jiwer is allowed only to verify your WER: compare('WER vs jiwer', mine, jiwer.wer(ref, hyp))

In [ ]:
# res.set('wer_sentence1_wav2vec2', value); res.set('wer_widest_beam', value)

## Part 7: Speech synthesis
TODO: [Critical] text normaliser, [Critical] Griffin-Lim, pretrained TTS (tts_synthesise), STOI (stoi_score).

In [ ]:
# [starter] plumbing, not counted as your own code
# Setup for Part 7 (plumbing only). The TTS model is fixed by your configuration.
tts_name = cfg['tts']['torchaudio_pipeline']
text0 = cfg['sentences_spoken'][0]
# Neural TTS takes about a minute per sentence on a CPU; uncomment when you are ready.
# wav_tts, fs_tts = tts_synthesise(text0, tts_name)
# import soundfile as sf; sf.write('figures/tts_sent0.wav', wav_tts, fs_tts)
# STOI (verification metric): stoi_score(reference, degraded, fs) with both signals at the same sampling rate
# [Critical] text normaliser and Griffin-Lim: yours. Spectral convergence after 30 iterations -> res.set('griffinlim_convergence_30', value)

In [ ]:
# res.set('griffinlim_convergence_30', value)

## Finish
Run `!python voicelab.py progress` for what is left. Then `python voicelab.py handwork` (photos in `handwork/`), `python voicelab.py report`, and `python voicelab.py package` once the notebook is run with outputs kept.